# 장미 잎 탐색 + 상태 판정 + VLM 검증 (Colab)

- **1단계 잎 탐색:** 기존 세그멘테이션 모델 `leaf_seg_v8r2.pt`
- **2단계 상태 분류:** 분류 모델 3개 중 선택 (4번 셀의 `CLS_MODEL`, 클래스는 모두 black_spot, downy_mildew, healthy, insects_infected)
- **3단계 VLM 검증 (선택):** 이상·의심 잎과 건강 잎 일부를 VLM(Claude 또는 Qwen2.5-VL)에게 다시 보여 주고 판정을 비교

**실행 순서**
1. 메뉴에서 **런타임 > 런타임 유형 변경 > T4 GPU**를 선택합니다.
2. `rose_leaf_colab_package.zip`을 Google Drive의 `내 드라이브/rose_leaf_colab(VLM적용+API방식변경)` 폴더에 올려 둡니다. 다른 폴더에 두려면 2번 셀의 `Path("/content/drive/MyDrive/rose_leaf_colab(VLM적용+API방식변경)")` 경로를 바꿉니다. zip을 찾지 못하면 2번 셀에서 직접 업로드하라고 안내합니다.
3. Claude로 VLM 검증(6번 단계)을 하려면 API 키를 Colab 보안 비밀에 미리 넣어 둡니다. 방법은 6번 단계의 **API 키 넣는 방법**에 있습니다.
4. 위에서부터 셀을 차례대로 실행합니다.

## 1. GPU 확인

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU로 실행되어 느립니다. 런타임 유형을 T4 GPU로 바꾸세요)")

## 2. 패키지 가져오기 및 설치

In [ ]:
import os, shutil, zipfile
from pathlib import Path
from google.colab import drive, files

USE_DRIVE = True  # Drive를 쓰지 않으려면 False
ZIP_NAME = "rose_leaf_colab_package.zip"
WORK = Path("/content/rose_leaf")

zip_path = None
if USE_DRIVE:
    drive.mount("/content/drive")
    p = Path("/content/drive/MyDrive/rose_leaf_colab(VLM적용+API방식변경)") / ZIP_NAME
    if p.exists():
        zip_path = p
    else:
        print("Drive에서 찾지 못했습니다:", p)
if zip_path is None:
    print(ZIP_NAME, "파일을 선택해 업로드하세요")
    zip_path = Path("/content") / next(iter(files.upload()))

shutil.rmtree(WORK, ignore_errors=True)
zipfile.ZipFile(zip_path).extractall(WORK)
print("압축 해제 완료:", sorted(x.name for x in WORK.iterdir()))

In [ ]:
!pip install -q -r /content/rose_leaf/requirements.txt

## 3. 사진 올리기

사진(jpg, png, bmp, webp)을 여러 장 선택할 수 있습니다. 아무것도 선택하지 않고 취소하면 포함된 샘플 사진으로 실행합니다.

In [ ]:
PHOTOS = Path("/content/photos")
shutil.rmtree(PHOTOS, ignore_errors=True)
PHOTOS.mkdir()

try:
    uploaded = files.upload()
except Exception:
    uploaded = {}
for name in uploaded:
    shutil.move(f"/content/{name}", PHOTOS / name)
if not uploaded:
    shutil.copy(WORK / "samples" / "fix_eafda190.jpg", PHOTOS)
    print("샘플 사진으로 실행합니다")
print(sorted(x.name for x in PHOTOS.iterdir()))

## 4. 실행

분류 모델을 `CLS_MODEL`에서 고릅니다. 잎 탐색(1단계) 모델은 같습니다.

| CLS_MODEL | 모델 | val 정확도 |
|---|---|---|
| `leaf_cls_yolov8m.pt` | YOLOv8m-cls (첫 번째 모델) | 0.957 |
| `leaf_cls_yolov8m_du.pt` | YOLOv8m-cls (du) | 0.950 |
| `leaf_cls_yolo26m_du.pt` | YOLO26m-cls (du) | 0.948 |

val 정확도는 각 모델 파일에 저장된 학습 기록 값입니다.

In [ ]:
CLS_MODEL = "leaf_cls_yolov8m.pt"  # "leaf_cls_yolov8m.pt" | "leaf_cls_yolov8m_du.pt" | "leaf_cls_yolo26m_du.pt"

RESULTS = Path("/content/results")
shutil.rmtree(RESULTS, ignore_errors=True)
!cd /content/rose_leaf && python run.py /content/photos --out /content/results --cls {CLS_MODEL}

## 5. 결과 보기

| 상태 | 의미 |
|---|---|
| FRESH | 건강 |
| ANOMALY:black_spot / downy_mildew / insects_infected | 이상 (클래스 표시) |
| UNKNOWN | 이상 의심, 관찰 대상 |
| HOLD_SHAPE | 잎이 합쳐졌거나 사진 가장자리에 잘림, 판정 안 함 |

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(RESULTS / "summary.csv"))
for f in sorted(RESULTS.glob("*_vis.jpg")):
    print(f.name)
    display(Image(filename=str(f), width=1000))

이상 또는 의심으로 판정된 잎 목록입니다.

In [ ]:
rows = []
for f in sorted(RESULTS.glob("*.json")):
    if f.stem.endswith("_vlm"):
        continue
    d = json.loads(f.read_text())
    for l in d["leaves"]:
        if l["state"] in ("ANOMALY", "UNKNOWN"):
            rows.append(dict(photo=d["photo"], id=l["id"], state=l["state"], cls=l["anomaly_class"], box=l["box"], **l["probs"]))
display(pd.DataFrame(rows) if rows else "이상 또는 의심 잎 없음")

## 6. VLM으로 검증 (선택)

분류 모델의 결과를 VLM에게 한 번 더 확인받습니다. 각 잎마다 **확대 사진**과 **잎 윤곽을 표시한 주변 사진** 두 장을 보내고,
분류 모델의 답은 알려 주지 않은 채(선입견 없이) 6가지 중 하나로 판정하게 한 뒤 결과를 비교합니다.

- 검증 대상: 이상(ANOMALY)·의심(UNKNOWN) 잎 전부 + 건강(FRESH) 잎 중 무작위 `FRESH_SAMPLE`장 (놓친 이상이 있는지 확인용)
- **Claude** (`VLM_BACKEND = "claude"`): Anthropic API 키가 필요하고 사용량만큼 요금이 나갑니다.
  키 넣는 방법은 바로 아래 **API 키 넣는 방법**에 있습니다.
- **Qwen2.5-VL-3B** (`VLM_BACKEND = "qwen"`): 무료 오픈 모델, T4 GPU에서 실행 (처음에 약 7GB 내려받음). Claude보다 판단력이 약합니다.
- `"auto"`: 키가 있으면 Claude, 없으면 Qwen

| 비교 결과 (verdict) | 의미 |
|---|---|
| AGREE | 분류 모델과 VLM 판정 일치 |
| FALSE_ALARM | 분류 모델은 이상, VLM은 건강 → 오경보 가능성 |
| MISSED | 분류 모델은 건강, VLM은 이상 → 놓친 이상 가능성 |
| OTHER_CLASS | 둘 다 이상이지만 종류가 다름 |
| VLM_HEALTHY / VLM_ANOMALY | 분류 모델이 의심(UNKNOWN)으로 남긴 잎에 대한 VLM 판정 |
| NOT_A_LEAF | 실제 잎이 아님 (거울 반사, 줄기, 여러 잎이 합쳐짐 등) |
| UNCLEAR | VLM도 판단 불가 |

### API 키 넣는 방법 (Claude를 쓸 때만)

이 노트북은 키를 **Colab 보안 비밀**에서 읽습니다. 코드에 키를 직접 적지 않습니다.

**1. API 키 발급**
1. Anthropic Console(console.anthropic.com)에 로그인합니다.
2. **API Keys** 메뉴에서 **Create Key**를 눌러 키를 만듭니다.
3. 만들어진 키(`sk-ant-`로 시작)를 복사합니다. 키는 만들 때 한 번만 보이므로 바로 복사해 둡니다.
4. 호출 요금은 Console의 **Billing**에 충전된 크레딧에서 나갑니다. 크레딧이 없으면 호출이 실패합니다.

**2. Colab 보안 비밀에 저장**
1. Colab 화면 왼쪽 세로 막대의 **열쇠 아이콘(보안 비밀)**을 누릅니다.
2. **새 보안 비밀 추가**를 누릅니다.
3. **이름**에 `ANTHROPIC_API_KEY`를 그대로 입력합니다 (대소문자, 밑줄까지 정확히).
4. **값**에 복사한 키를 붙여 넣습니다. 앞뒤에 공백이 들어가지 않게 합니다.
5. 그 줄의 **노트북 액세스** 스위치를 켭니다.

**3. 확인**
1. 아래 셀을 실행합니다. 처음 실행할 때 보안 비밀 액세스를 허용할지 묻는 창이 뜨면 허용합니다.
2. `ANTHROPIC_API_KEY를 불러왔습니다`가 나오면 성공입니다. 이어서 `--dry-run` 결과에 `backend claude`가 나옵니다.
3. `보안 비밀에 ANTHROPIC_API_KEY가 없습니다`가 나오면 이름 철자와 **노트북 액세스** 스위치를 확인합니다.
   `VLM_BACKEND = "auto"`이면 이 경우 Qwen으로 실행됩니다.

**알아 둘 점**
- 보안 비밀은 Google 계정에 저장되어 다른 노트북에서도 다시 쓸 수 있습니다. 다만 노트북마다 **노트북 액세스**를 켜야 합니다.
- 노트북을 공유해도 보안 비밀 값은 함께 공유되지 않습니다.
- 키를 코드 셀에 직접 붙여 넣지 마세요. 노트북을 저장하거나 공유할 때 키가 함께 남습니다.
- 키가 유출됐다고 생각되면 Anthropic Console에서 그 키를 바로 삭제하고 새로 만듭니다.
- 보안 비밀 대신 파일(`ANTHROPIC_API_KEY`)로 키를 넣으려면 `rose_leaf_colab_keyfile.ipynb`를 쓰세요.

In [ ]:
VLM_BACKEND = "auto"               # "auto" | "claude" | "qwen"
CLAUDE_MODEL = "claude-opus-5-5"
QWEN_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
FRESH_SAMPLE = 5                   # 사진마다 검증할 건강 잎 수
MAX_PER_PHOTO = 40                 # 사진마다 검증할 최대 잎 수 (비용 상한)

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("ANTHROPIC_API_KEY를 불러왔습니다")
except Exception:
    print("보안 비밀에 ANTHROPIC_API_KEY가 없습니다. auto로 두면 Qwen으로 실행합니다")

VLM_MODEL = CLAUDE_MODEL if VLM_BACKEND == "claude" or (VLM_BACKEND == "auto" and os.environ.get("ANTHROPIC_API_KEY")) else QWEN_MODEL
!cd /content/rose_leaf && python verify.py --photos /content/photos --results /content/results --backend {VLM_BACKEND} --model {VLM_MODEL} --fresh {FRESH_SAMPLE} --max {MAX_PER_PHOTO} --dry-run

위에 나온 검증할 잎 수를 확인한 뒤 실행합니다. Claude는 잎 하나에 요청 한 번이 나갑니다.

In [ ]:
!cd /content/rose_leaf && python verify.py --photos /content/photos --results /content/results --backend {VLM_BACKEND} --model {VLM_MODEL} --fresh {FRESH_SAMPLE} --max {MAX_PER_PHOTO}

In [ ]:
review = pd.read_csv(RESULTS / "vlm_review.csv")
display(review.groupby(["photo", "verdict"]).size().unstack(fill_value=0))
display(review[["photo", "cls_model", "id", "model", "model_p", "vlm_label", "confidence", "verdict", "symptoms", "reason", "error"]])
for f in sorted(RESULTS.glob("*_vlm.jpg")):
    print(f.name)
    display(Image(filename=str(f), width=1000))

## 7. 결과 내려받기

In [ ]:
shutil.make_archive("/content/results", "zip", RESULTS)
files.download("/content/results.zip")